## Model Creation and Evaluation

In [5]:
# import packages 
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

df = pd.read_csv('../data/cleaned_data.csv')

In [6]:
X = pd.get_dummies(df[['state', 'Event Type', 'county', 'duration']], drop_first=True)
y = df['hour']

# Split before scaling to avoid data leakage
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create scaled copies (for models sensitive to feature magnitude)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_unscaled)
X_test_scaled = scaler.transform(X_test_unscaled)

# Convert back to DataFrame for convenience (especially with SHAP or plotting)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_unscaled.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_unscaled.index)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [7]:
model = LinearRegression()

# Cross validation
cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = model.predict(X_test_scaled)

# Metrics
r2 = r2_score(y_test, y_pred_lr)
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Linear Regression Results
CV R2: 0.13497305202501958
Test R2: 0.1465522868962683
MAE (Hours): 5.557576010902281
RMSE (Hours): 6.738600224986026


These results indicate that the model explains ~14% of the outage timing and is off by roughly 5-7 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

In [9]:
X_train = X_train_unscaled
X_test = X_test_unscaled

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

cv_r2 = cross_val_score(model, X_train, y_train, cv=10, scoring='r2').mean()

model.fit(X_train, y_train)

y_pred_rf = model.predict(X_test)

r2 = r2_score(y_test, y_pred_rf)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Random Forest Results
CV R2: 0.5421835417281884
Test R2: 0.5470888179022608
MAE (Hours): 2.875209179683794
RMSE (Hours): 4.908940342971553


These results indicate that the model explains ~54% of the outage timing and is off by roughly 3-5 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

In [10]:
X_train = X_train_unscaled
X_test = X_test_unscaled

model = GradientBoostingRegressor(n_estimators=100, random_state=42)

cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=10, scoring='r2').mean()

model.fit(X_train_scaled, y_train)

y_pred_gb = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred_gb)
mae = mean_absolute_error(y_test, y_pred_gb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print("Gradient Boosting Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Gradient Boosting Results
CV R2: 0.13625539952004098
Test R2: 0.13396375136734184
MAE (Hours): 5.790785074525885
RMSE (Hours): 6.788116194837465


These results indicate that the model explains ~13% of the outage timing and is off by roughly 6-8 hours. The CV R2 and Test R2 being similar numbers indicates that the model is not overfitting. 

## Hyperparameter Tuning

In [11]:
# Parameter distribution
lr_param_dist = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Initialize model
lr = LinearRegression()

# Randomized Search
lr_rand = RandomizedSearchCV(
    lr,
    lr_param_dist,
    n_iter=4,      # number of random combinations to try (max 4 here)
    cv=3,          # fewer CV folds for speed
    scoring='r2',
    n_jobs=-1,
    random_state=42,
)

# Fit the model
lr_rand.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_tuned = lr_rand.best_estimator_.predict(X_test_scaled)

# Metrics
print("Test R2:", r2_score(y_test, y_pred_lr_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_lr_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr_tuned)))

Test R2: 0.1465522868962683
MAE: 5.557576010902281
RMSE: 6.738600224986026


These results indicate that the model explains ~14% of the outage timing and is off by roughly 5-7 hours. We can see that even after hyperparamter tuning, the results did not change. So we can conclude that the logistic regression model is not an effective model for forecasting time of day of an outage. 

In [14]:
# Parameter distribution
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize model
rf = RandomForestRegressor(random_state=42)

# Randomized Search
rf_random = RandomizedSearchCV(
    rf,
    rf_param_grid,
    n_iter=4,      # number of random combinations to try
    cv=3,           # fewer folds for speed
    scoring='r2',
    n_jobs=-1,      # parallelize across CPUs
    random_state=42
)

# Fit the model
rf_random.fit(X_train_scaled, y_train)

# Use the best model for predictions
y_pred_rf_tuned = rf_random.best_estimator_.predict(X_test_scaled)

# Metrics
print("Test R2:", r2_score(y_test, y_pred_rf_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)))

Test R2: 0.5587821297017046
MAE: 2.953014516027125
RMSE: 4.8451561673113375


These results indicate that the model explains ~56% of the outage timing and is off by roughly 3-5 hours. This is slightly better than before fine tuning, but barely.

In [16]:
# Parameter distribution
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

# Initialize model
gb = GradientBoostingRegressor(random_state=42)

# Randomized Search
gb_random = RandomizedSearchCV(
    gb,
    gb_param_grid,
    n_iter=5,       # number of random combinations to try
    cv=5,            # fewer folds for speed
    scoring='r2',
    n_jobs=-1,       # parallelize across CPUs
    random_state=42
)

# Fit the model
gb_random.fit(X_train_scaled, y_train)

# Use the best model for predictions
y_pred_gb_tuned = gb_random.best_estimator_.predict(X_test_scaled)

# Metrics
print("Test R2:", r2_score(y_test, y_pred_gb_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_gb_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb_tuned)))

Test R2: 0.2786054485937336
MAE: 5.167683061071844
RMSE: 6.19537589322059


These results indicate that the model explains ~28% of the outage timing and is off by roughly 3-5 hours. This R2 score is over double than before hyperparameter tuning, but overall, still a low score. 

In [19]:
#### import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Assign predictions from your code
y_pred_lr_before = y_pred_lr  # after first Linear Regression run
y_pred_rf_before = y_pred_rf  # after first Random Forest run
y_pred_gb_before = y_pred_gb # after first Gradient Boosting run

y_pred_lr_after = y_pred_lr_tuned
y_pred_rf_after = y_pred_rf_tuned
y_pred_gb_after = y_pred_gb_tuned

models = ["Linear Regression", "Random Forest", "Gradient Boosting"]
rows = []

for model, y_before, y_after in zip(
    models,
    [y_pred_lr_before, y_pred_rf_before, y_pred_gb_before],
    [y_pred_lr_after, y_pred_rf_after, y_pred_gb_after]
):
    rows.append({
        "Model": model,
        "R2 Before": r2_score(y_test, y_before),
        "R2 After": r2_score(y_test, y_after),
        "MAE Before": mean_absolute_error(y_test, y_before),
        "MAE After": mean_absolute_error(y_test, y_after),
        "RMSE Before": np.sqrt(mean_squared_error(y_test, y_before)),
        "RMSE After": np.sqrt(mean_squared_error(y_test, y_after))
    })

performance_table = pd.DataFrame(rows)
print(performance_table)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 2))  
ax.axis('tight')
ax.axis('off')

# Create a table in the plot
table = ax.table(cellText=performance_table.values,
                 colLabels=performance_table.columns,
                 cellLoc='center',
                 loc='center')

plt.savefig("../outputs/performance_table.png", dpi=150)
plt.close()

               Model  R2 Before  R2 After  MAE Before  MAE After  RMSE Before  \
0  Linear Regression   0.146552  0.146552    5.557576   5.557576     6.738600   
1      Random Forest   0.547089  0.558782    2.875209   2.953015     4.908940   
2  Gradient Boosting   0.133964  0.278605    5.790785   5.167683     6.788116   

   RMSE After  
0    6.738600  
1    4.845156  
2    6.195376  
